In [0]:

from pyspark.sql import functions as F

CATALOG = "food_delivery_catalog"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

VALID_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.orders_valid"
SILVER_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.orders_cleaned"

# Read validated Bronze records
df_bronze_valid = spark.table(VALID_TABLE)

print("BRONZE VALID COUNT:", df_bronze_valid.count())

display(df_bronze_valid.limit(10))


BRONZE VALID COUNT: 83


order_id,order_date,customer_id,restaurant_name,city,food_category,quantity,item_price,delivery_fee,discount,order_status,batch_id,source_file,ingestion_timestamp
ORD0001,2026-10-05,C001,Restaurant 2,Mumbai,CHINESE,2.0,375.0,90.0,20.0,DELIVERED,BATCH_001,food_delivery_initial_100.csv,2026-11-26T05:10:44.030Z
ORD0002,2026-10-06,C002,Restaurant 3,Kochi,CONTINENTAL,3.0,550.0,130.0,40.0,CANCELLED,BATCH_001,food_delivery_initial_100.csv,2026-11-26T05:10:44.030Z
ORD0003,2026-10-07,C003,Restaurant 4,Delhi,INDIAN,4.0,725.0,170.0,60.0,PLACED,BATCH_001,food_delivery_initial_100.csv,2026-11-26T05:10:44.030Z
ORD0004,2026-10-08,C004,Restaurant 5,Coimbatore,CHINESE,5.0,900.0,210.0,0.0,DELIVERED,BATCH_001,food_delivery_initial_100.csv,2026-11-26T05:10:44.030Z
ORD0005,2026-10-09,C005,Restaurant 6,Bengaluru,CONTINENTAL,1.0,1075.0,250.0,20.0,CANCELLED,BATCH_001,food_delivery_initial_100.csv,2026-11-26T05:10:44.030Z
ORD0006,2026-10-10,C006,Restaurant 7,Chennai,INDIAN,2.0,1250.0,290.0,40.0,PLACED,BATCH_001,food_delivery_initial_100.csv,2026-11-26T05:10:44.030Z
ORD0007,2026-10-11,C007,Restaurant 8,Hyderabad,CHINESE,3.0,1425.0,50.0,60.0,DELIVERED,BATCH_001,food_delivery_initial_100.csv,2026-11-26T05:10:44.030Z
ORD0008,2026-10-12,C008,Restaurant 1,Pune,CONTINENTAL,4.0,200.0,90.0,0.0,CANCELLED,BATCH_001,food_delivery_initial_100.csv,2026-11-26T05:10:44.030Z
ORD0009,2026-10-13,C009,Restaurant 2,Mumbai,INDIAN,5.0,375.0,130.0,20.0,PLACED,BATCH_001,food_delivery_initial_100.csv,2026-11-26T05:10:44.030Z
ORD0010,2026-10-14,C010,Restaurant 3,Kochi,CHINESE,1.0,550.0,170.0,40.0,DELIVERED,BATCH_001,food_delivery_initial_100.csv,2026-11-26T05:10:44.030Z


In [0]:

# Clean and standardize Silver data
df_silver = (
    df_bronze_valid

    # Trim unwanted spaces
    .withColumn("order_id", F.trim("order_id"))
    .withColumn("customer_id", F.trim("customer_id"))
    .withColumn("restaurant_name", F.initcap(F.trim("restaurant_name")))
    .withColumn("city", F.initcap(F.trim("city")))
    .withColumn("food_category", F.upper(F.trim("food_category")))
    .withColumn("order_status", F.upper(F.trim("order_status")))

    # Convert data types
    .withColumn("order_date", F.to_date("order_date"))
    .withColumn("quantity", F.col("quantity").cast("DOUBLE").cast("INT"))
    .withColumn("item_price", F.col("item_price").cast("DECIMAL(10,2)"))
    .withColumn("delivery_fee", F.col("delivery_fee").cast("DECIMAL(10,2)"))
    .withColumn("discount", F.col("discount").cast("DECIMAL(10,2)"))

    # Calculate order amount
    .withColumn(
        "order_amount",
        (
            F.col("quantity") * F.col("item_price")
            + F.col("delivery_fee")
            - F.col("discount")
        ).cast("DECIMAL(12,2)")
    )
)

display(df_silver)


order_id,order_date,customer_id,restaurant_name,city,food_category,quantity,item_price,delivery_fee,discount,order_status,batch_id,source_file,ingestion_timestamp,order_amount
ORD0001,2026-10-05,C001,Restaurant 2,Mumbai,CHINESE,2,375.00,90.00,20.00,DELIVERED,BATCH_001,food_delivery_initial_100.csv,2026-11-26T05:10:44.030Z,820.00
ORD0002,2026-10-06,C002,Restaurant 3,Kochi,CONTINENTAL,3,550.00,130.00,40.00,CANCELLED,BATCH_001,food_delivery_initial_100.csv,2026-11-26T05:10:44.030Z,1740.00
ORD0003,2026-10-07,C003,Restaurant 4,Delhi,INDIAN,4,725.00,170.00,60.00,PLACED,BATCH_001,food_delivery_initial_100.csv,2026-11-26T05:10:44.030Z,3010.00
ORD0004,2026-10-08,C004,Restaurant 5,Coimbatore,CHINESE,5,900.00,210.00,0.00,DELIVERED,BATCH_001,food_delivery_initial_100.csv,2026-11-26T05:10:44.030Z,4710.00
ORD0005,2026-10-09,C005,Restaurant 6,Bengaluru,CONTINENTAL,1,1075.00,250.00,20.00,CANCELLED,BATCH_001,food_delivery_initial_100.csv,2026-11-26T05:10:44.030Z,1305.00
ORD0006,2026-10-10,C006,Restaurant 7,Chennai,INDIAN,2,1250.00,290.00,40.00,PLACED,BATCH_001,food_delivery_initial_100.csv,2026-11-26T05:10:44.030Z,2750.00
ORD0007,2026-10-11,C007,Restaurant 8,Hyderabad,CHINESE,3,1425.00,50.00,60.00,DELIVERED,BATCH_001,food_delivery_initial_100.csv,2026-11-26T05:10:44.030Z,4265.00
ORD0008,2026-10-12,C008,Restaurant 1,Pune,CONTINENTAL,4,200.00,90.00,0.00,CANCELLED,BATCH_001,food_delivery_initial_100.csv,2026-11-26T05:10:44.030Z,890.00
ORD0009,2026-10-13,C009,Restaurant 2,Mumbai,INDIAN,5,375.00,130.00,20.00,PLACED,BATCH_001,food_delivery_initial_100.csv,2026-11-26T05:10:44.030Z,1985.00
ORD0010,2026-10-14,C010,Restaurant 3,Kochi,CHINESE,1,550.00,170.00,40.00,DELIVERED,BATCH_001,food_delivery_initial_100.csv,2026-11-26T05:10:44.030Z,680.00


In [0]:

(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

print("Silver table created successfully!")
print("SILVER COUNT:", spark.table(SILVER_TABLE).count())


Silver table created successfully!
SILVER COUNT: 83


In [0]:
%sql
SELECT *
FROM food_delivery_catalog.silver.orders_cleaned
LIMIT 10;

order_id,order_date,customer_id,restaurant_name,city,food_category,quantity,item_price,delivery_fee,discount,order_status,batch_id,source_file,ingestion_timestamp,order_amount
ORD0001,2026-10-05,C001,Restaurant 2,Mumbai,CHINESE,2,375.00,90.00,20.00,DELIVERED,BATCH_001,food_delivery_initial_100.csv,2026-11-26T05:10:44.030Z,820.00
ORD0002,2026-10-06,C002,Restaurant 3,Kochi,CONTINENTAL,3,550.00,130.00,40.00,CANCELLED,BATCH_001,food_delivery_initial_100.csv,2026-11-26T05:10:44.030Z,1740.00
ORD0003,2026-10-07,C003,Restaurant 4,Delhi,INDIAN,4,725.00,170.00,60.00,PLACED,BATCH_001,food_delivery_initial_100.csv,2026-11-26T05:10:44.030Z,3010.00
ORD0004,2026-10-08,C004,Restaurant 5,Coimbatore,CHINESE,5,900.00,210.00,0.00,DELIVERED,BATCH_001,food_delivery_initial_100.csv,2026-11-26T05:10:44.030Z,4710.00
ORD0005,2026-10-09,C005,Restaurant 6,Bengaluru,CONTINENTAL,1,1075.00,250.00,20.00,CANCELLED,BATCH_001,food_delivery_initial_100.csv,2026-11-26T05:10:44.030Z,1305.00
ORD0006,2026-10-10,C006,Restaurant 7,Chennai,INDIAN,2,1250.00,290.00,40.00,PLACED,BATCH_001,food_delivery_initial_100.csv,2026-11-26T05:10:44.030Z,2750.00
ORD0007,2026-10-11,C007,Restaurant 8,Hyderabad,CHINESE,3,1425.00,50.00,60.00,DELIVERED,BATCH_001,food_delivery_initial_100.csv,2026-11-26T05:10:44.030Z,4265.00
ORD0008,2026-10-12,C008,Restaurant 1,Pune,CONTINENTAL,4,200.00,90.00,0.00,CANCELLED,BATCH_001,food_delivery_initial_100.csv,2026-11-26T05:10:44.030Z,890.00
ORD0009,2026-10-13,C009,Restaurant 2,Mumbai,INDIAN,5,375.00,130.00,20.00,PLACED,BATCH_001,food_delivery_initial_100.csv,2026-11-26T05:10:44.030Z,1985.00
ORD0010,2026-10-14,C010,Restaurant 3,Kochi,CHINESE,1,550.00,170.00,40.00,DELIVERED,BATCH_001,food_delivery_initial_100.csv,2026-11-26T05:10:44.030Z,680.00


In [0]:
%sql
SELECT
    COUNT(*) AS total_orders,
    ROUND(SUM(order_amount), 2) AS total_revenue,
    ROUND(AVG(order_amount), 2) AS average_order_amount
FROM food_delivery_catalog.silver.orders_cleaned;

total_orders,total_revenue,average_order_amount
83,159225.00,1918.37
